# Python Assignment 17 :Time series

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("exchange_rate.csv")
df.head()

In [ ]:
df["date"] = pd.to_datetime(df["date"], format="%d-%m-%Y %H:%M")

In [ ]:
df.info()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df["date"], df["Ex_rate"], label="Exchange rate")
plt.title("Exchange Rate Over Time")
plt.xlabel("Date")
plt.ylabel("Exchange Rate")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
print(df.isnull().sum())

In [ ]:
# Forward-fill missing values in time order
df = df.sort_values("date").copy()
df["Ex_rate"] = df["Ex_rate"].ffill()
print("Missing values after filling:", df["Ex_rate"].isnull().sum())

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=df["Ex_rate"])
plt.title("Exchange Rate Boxplot for Outlier Inspection")
plt.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(df["Ex_rate"].dropna())
print("ADF p-value before differencing:", result[1])

In [ ]:
series_diff = df["Ex_rate"].diff().dropna()
result = adfuller(series_diff)
print("ADF p-value after first differencing:", result[1])

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(series_diff, lags=30, ax=axes[0])
axes[0].set_title("ACF of Differenced Series")
plot_pacf(series_diff, lags=30, ax=axes[1], method="ywm")
axes[1].set_title("PACF of Differenced Series")
plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

# Fit ARIMA on the full series for model summary and residual diagnostics
full_model = ARIMA(df["Ex_rate"], order=(1, 1, 1))
full_model_fit = full_model.fit()
print(full_model_fit.summary())

In [ ]:
residuals = full_model_fit.resid

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
residuals.plot(ax=axes[0], title="ARIMA Residuals")
residuals.plot(kind="kde", ax=axes[1], title="Residual Density")
plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

plot_acf(residuals, lags=30)
plt.title("ACF of ARIMA Residuals")
plt.tight_layout()
plt.show()

## Train-test split and forecasting

In [ ]:
split_idx = int(len(df) * 0.8)
train = df["Ex_rate"].iloc[:split_idx]
test = df["Ex_rate"].iloc[split_idx:]
print("Train size:", len(train))
print("Test size:", len(test))

In [ ]:
model = ARIMA(train, order=(1, 1, 1))
model_fit = model.fit()

In [ ]:
forecast = model_fit.forecast(steps=len(test))
forecast = pd.Series(np.asarray(forecast), index=test.index, name="ARIMA")

In [ ]:
train_dates = df["date"].iloc[:split_idx]
test_dates = df["date"].iloc[split_idx:]

plt.figure(figsize=(12, 5))
plt.plot(train_dates, train, label="Train")
plt.plot(test_dates, test, label="Actual test")
plt.plot(test_dates, forecast, label="ARIMA forecast")
plt.title("ARIMA Forecast vs Actual")
plt.xlabel("Date")
plt.ylabel("Exchange Rate")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Recursive moving-average forecast: do not use actual test values to create predictions
window = 5
history = list(train.dropna().iloc[-window:])
ma_predictions = []

for _ in range(len(test)):
    next_prediction = float(np.mean(history[-window:]))
    ma_predictions.append(next_prediction)
    history.append(next_prediction)

forecast_model2 = pd.Series(ma_predictions, index=test.index, name="Moving Average")

## Exponential Smoothing forecast

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

exp_model = ExponentialSmoothing(
    train,
    trend="add",
    seasonal=None,
    initialization_method="estimated"
)
exp_model_fit = exp_model.fit()
forecast_exp = exp_model_fit.forecast(steps=len(test))
forecast_exp = pd.Series(np.asarray(forecast_exp), index=test.index, name="Exponential Smoothing")

## Compare forecast accuracy

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate_forecast(actual, predicted, model_name):
    actual_values = np.asarray(actual, dtype=float)
    predicted_values = np.asarray(predicted, dtype=float)
    mae_value = mean_absolute_error(actual_values, predicted_values)
    rmse_value = np.sqrt(mean_squared_error(actual_values, predicted_values))
    denominator = np.maximum(np.abs(actual_values), 1e-8)
    mape_value = np.mean(np.abs((actual_values - predicted_values) / denominator)) * 100
    print(f"{model_name} -> MAE: {mae_value:.6f}, RMSE: {rmse_value:.6f}, MAPE: {mape_value:.4f}%")
    return mae_value, rmse_value, mape_value

mae_arima, rmse_arima, mape_arima = evaluate_forecast(test, forecast, "ARIMA")
mae_m2, rmse_m2, mape_m2 = evaluate_forecast(test, forecast_model2, "Moving Average")
mae_exp, rmse_exp, mape_exp = evaluate_forecast(test, forecast_exp, "Exponential Smoothing")

In [ ]:
metrics_df = pd.DataFrame({
    "Model": ["ARIMA", "Moving Average", "Exponential Smoothing"],
    "MAE": [mae_arima, mae_m2, mae_exp],
    "RMSE": [rmse_arima, rmse_m2, rmse_exp],
    "MAPE (%)": [mape_arima, mape_m2, mape_exp]
})

print("Forecast evaluation on the held-out test set:")
display(metrics_df.round(6))

for metric in ["MAE", "RMSE", "MAPE (%)"]:
    best_row = metrics_df.loc[metrics_df[metric].idxmin()]
    print(f"Lowest {metric}: {best_row['Model']} ({best_row[metric]:.6f})")

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(test_dates, test, label="Actual")
plt.plot(test_dates, forecast, label="ARIMA")
plt.plot(test_dates, forecast_model2, label="Moving Average")
plt.plot(test_dates, forecast_exp, label="Exponential Smoothing")
plt.title("Forecast Model Comparison")
plt.xlabel("Date")
plt.ylabel("Exchange Rate")
plt.legend()
plt.tight_layout()
plt.show()

### Conclusion

In this time series forecasting assignment, ARIMA, Moving Average, and Exponential Smoothing models were implemented and evaluated using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and Mean Absolute Percentage Error (MAPE).

Based on the evaluation results, **ARIMA performed best among the three models**, achieving the lowest MAE of 0.177710, RMSE of 0.205437, and MAPE of 22.797966%. The Moving Average model produced very similar results, while Exponential Smoothing recorded higher errors.

Therefore, ARIMA is the preferred model among those evaluated for forecasting the exchange rate in this experiment. However, its forecasting accuracy could potentially be improved by testing alternative ARIMA parameters and comparing additional models.
